# Simple: Fidelity Metrics with PAMOLA.CORE

**Goal**: Learn fidelity metric calculation basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load original and transformed data from examples/data_examples/
- Calculate fidelity metrics (KS test, KL divergence) using execute()
- Compare distribution similarity between datasets
- Understand statistical significance and effect sizes

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── metrics/
        └── 01_fidelity_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.metrics.operations.fidelity_ops import FidelityOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Load original and transformed data from `examples/data_examples/`
- Auto-create sample datasets if files don't exist
- Inspect both dataset structures

**Expected output:** Two DataFrames representing original and anonymized versions

In [ ]:
examples_dir = project_root / 'examples'
original_path = examples_dir / 'data_examples' / 'sample.csv'
transformed_path = examples_dir / 'data_examples' / 'sample.csv'

# Create original data if it doesn't exist
if not original_path.exists():
    print("⚠️  Original file not found, creating sample data...")
    original_path.parent.mkdir(parents=True, exist_ok=True)
    
    np.random.seed(42)
    original_data = pd.DataFrame({
        'record_id': range(1, 101),
        'age': np.random.normal(35, 10, 100).astype(int),
        'income': np.random.normal(60000, 20000, 100).astype(int),
        'credit_score': np.random.normal(700, 50, 100).astype(int),
        'category': np.random.choice(['A', 'B', 'C', 'D'], 100, p=[0.4, 0.3, 0.2, 0.1])
    })
    original_data.to_csv(original_path, index=False)
    print(f"✓ Original data created")

# Create transformed data if it doesn't exist (simulated anonymization)
if not transformed_path.exists():
    print("⚠️  Transformed file not found, creating sample data...")
    transformed_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Read original to create transformed version
    original_temp = pd.read_csv(original_path)
    
    np.random.seed(43)
    transformed_data = pd.DataFrame({
        'record_id': range(1, 101),
        # Add slight noise to simulate anonymization
        'age': (original_temp['age'] + np.random.normal(0, 2, 100)).astype(int),
        'income': (original_temp['income'] + np.random.normal(0, 3000, 100)).astype(int),
        'credit_score': (original_temp['credit_score'] + np.random.normal(0, 10, 100)).astype(int),
        # Generalize categories slightly
        'category': np.random.choice(['A', 'B', 'C', 'OTHER'], 100, p=[0.38, 0.32, 0.22, 0.08])
    })
    transformed_data.to_csv(transformed_path, index=False)
    print(f"✓ Transformed data created")

# Load both datasets
original_df = read_csv(original_path)
transformed_df = read_csv(transformed_path)

print(f"\n📊 ORIGINAL Dataset: {len(original_df)} records, {len(original_df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(original_df.head())

print(f"\n📊 TRANSFORMED Dataset: {len(transformed_df)} records, {len(transformed_df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(transformed_df.head())

# Show distribution comparison
print(f"\n📈 Distribution Comparison:")
for col in ['age', 'income', 'current_salary_cad', 'years_of_experience']:
    if col not in original_df.columns or col not in transformed_df.columns:
        print(f"\n  {col}: ⚠️ column not found, skipped")
        continue

    orig_mean = original_df[col].mean()
    trans_mean = transformed_df[col].mean()
    orig_std = original_df[col].std()
    trans_std = transformed_df[col].std()

    print(f"\n  {col}:")
    print(f"    Original:    mean={orig_mean:.2f}, std={orig_std:.2f}")
    print(f"    Transformed: mean={trans_mean:.2f}, std={trans_std:.2f}")
    print(
        f"    Difference:  "
        f"mean={abs(orig_mean - trans_mean):.2f}, "
        f"std={abs(orig_std - trans_std):.2f}"
    )

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with both DataFrames
- Configure progress tracker
- **Configure fidelity metrics**

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "original_dataset_name": "original_dataset",
        "transformed_dataset_name": "transformed_dataset",
        "fidelity_metrics": ["ks", "kl"]
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "original_dataset_name": "original_dataset",
        "transformed_dataset_name": "transformed_dataset",
        "fidelity_metrics": ["ks", "kl"],  # Kolmogorov-Smirnov, KL Divergence
        "columns": ["years_of_experience", "current_salary_cad"],  # Numeric columns to compare
        "metric_params": {
            "kl": {
                "key_fields": ["years_of_experience"],
                "value_field": "current_salary_cad",
                "aggregation": "count",
                "epsilon": 0.01,
                "confidence_level": 0.95,
                "normalize": True
            },
            "ks": {
                "key_fields": ["years_of_experience"],
                "value_field": "current_salary_cad",
                "aggregation": "sum",
                "confidence_level": 0.95,
                "normalize": True
            }
        }
    }

kwargs = create_config_kwargs()
fidelity_metrics = kwargs['fidelity_metrics']
columns = kwargs['columns']
metric_params = kwargs['metric_params']
original_dataset_name = kwargs['original_dataset_name']
transformed_dataset_name = kwargs['transformed_dataset_name']

print(f"\n🔍 Validating configuration...\n")
print(f"✓ Original dataset: '{original_dataset_name}'")
print(f"✓ Transformed dataset: '{transformed_dataset_name}'")
print(f"✓ Fidelity metrics: {', '.join(fidelity_metrics)}")
print(f"✓ Columns to compare: {', '.join(columns)}")
print(f"✓ Metric params: {', '.join(metric_params)}") 

# Validate columns exist in both datasets
for col in columns:
    if col not in original_df.columns:
        raise ValueError(f"❌ Column '{col}' not found in original dataset!")
    if col not in transformed_df.columns:
        raise ValueError(f"❌ Column '{col}' not found in transformed dataset!")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="metrics",
    description="Fidelity metrics calculation between original and transformed data",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {
    "original_dataset_name": original_dataset_name,
    "transformed_dataset_name": transformed_dataset_name
}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource with both datasets
data_source = DataSource(
    dataframes={
        "original_dataset": original_df,
        "transformed_dataset": transformed_df
    }
)
print("✓ DataSource created with 2 datasets")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=5,
    description="Calculating fidelity metrics",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create FidelityOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `fidelity_metrics`: List of metrics to calculate ['ks', 'kl']
- `metric_params`: Additional parameters to customize metric computation.
- `columns`: Columns to compare between datasets
- `normalize=True`: Normalize data before comparison
- `confidence_level=0.95`: Statistical confidence level (95%)
- `generate_visualization=True`: Create charts
- `save_output=False`: Metrics operation doesn't save output data

**Fidelity Metrics:**
- **KS (Kolmogorov-Smirnov)**: Tests if distributions differ significantly
- **KL (Kullback-Leibler)**: Measures divergence between distributions

**Interpretation:**
- Lower values = More similar distributions = Better fidelity
- Higher p-values (KS) = Distributions not significantly different

In [ ]:
# Create operation with production-style configuration
operation = FidelityOperation(
    # Core parameters
    fidelity_metrics=fidelity_metrics,
    metric_params=metric_params,
    columns=columns,
    
    # Statistical parameters
    normalize=True,
    confidence_level=0.95,
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=False,  # Metrics don't save output data
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Fidelity metrics:          {', '.join(operation.fidelity_metrics)}")
print(f"  Metrics params:          {', '.join(operation.metric_params)}")
print(f"  Columns:          {', '.join(operation.columns)}")
print(f"  Normalize:        {operation.normalize}")
print(f"  Confidence:       {operation.confidence_level}")
print(f"  Visualizations:   {operation.generate_visualization}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing fidelity metrics calculation...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the metrics from task_dir
- Review statistical measures
- Interpret fidelity scores

**Generated artifacts:**
- Metrics JSON in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load metrics file
metrics_files = list(task_dir.glob('metrics/*.json'))
if metrics_files:
    # Sort by modification time and get latest
    metrics_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest_metrics = metrics_files[0]
    
    with open(latest_metrics, 'r') as f:
        metrics_data = json.load(f)
    
    metrics = metrics_data.get('metrics', {})
    
    print("\n" + "=" * 80)
    print("📊 FIDELITY METRICS RESULTS")
    print("=" * 80)
    
    # Display KS Test Results
    if 'ks' in metrics:
        ks = metrics['ks']
        print("\n📈 Kolmogorov-Smirnov (KS) Test:")
        print(f"  KS Statistic:              {ks.get('ks_statistic', 'N/A')}")
        print(f"  P-Value:                   {ks.get('p_value', 'N/A')}")
        print(f"  Effect Size:               {ks.get('effect_size', 'N/A')}")
        print(f"  Statistical Significance:  {ks.get('statistical_significance', 'N/A')}")
        print(f"  Interpretation:            {ks.get('interpretation', 'N/A')}")
        if 'confidence_interval' in ks:
            ci = ks['confidence_interval']
            # Check if CI values are numeric before formatting
            if isinstance(ci[0], (int, float)) and isinstance(ci[1], (int, float)):
                print(f"  Confidence Interval:       [{ci[0]:.4f}, {ci[1]:.4f}]")
            else:
                print(f"  Confidence Interval:       [{ci[0]}, {ci[1]}]")

    # Display KL Divergence Results
    if 'kl' in metrics:
        kl = metrics['kl']
        print("\n📉 Kullback-Leibler (KL) Divergence:")
        print(f"  KL Divergence (nats):      {kl.get('kl_divergence', 'N/A')}")
        print(f"  KL Divergence (bits):      {kl.get('kl_divergence_bits', 'N/A')}")
        print(f"  Effect Size:               {kl.get('effect_size', 'N/A')}")
        print(f"  Statistical Significance:  {kl.get('statistical_significance', 'N/A')}")
        print(f"  JS Distance:               {kl.get('jensen_shannon_distance', 'N/A')}")
        print(f"  Interpretation:            {kl.get('interpretation', 'N/A')}")
        if 'confidence_interval' in kl:
            ci = kl['confidence_interval']
            # Check if CI values are numeric before formatting
            if isinstance(ci[0], (int, float)) and isinstance(ci[1], (int, float)):
                print(f"  Confidence Interval:       [{ci[0]:.4f}, {ci[1]:.4f}]")
            else:
                print(f"  Confidence Interval:       [{ci[0]}, {ci[1]}]")
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Datasets compared:  Original vs Transformed")
    print(f"  Records compared:   {len(original_df)}")
    print(f"  Columns compared:   {', '.join(columns)}")
    
    # Overall assessment
    print("\n💡 Overall Assessment:")
    if 'ks' in metrics and metrics['ks'].get('p_value', 0) > 0.05:
        print("  ✓ Distributions are statistically similar (KS p-value > 0.05)")
    if 'kl' in metrics and metrics['kl'].get('kl_divergence', 1.0) < 0.1:
        print("  ✓ Low divergence between distributions (KL < 0.1)")
    print("  → High fidelity: Transformed data preserves original distribution well")
    
else:
    print("⚠️  No metrics files found in", task_dir / 'metrics')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Complete fidelity metrics with statistical details
2. **Visualizations**: Bar charts showing KS and KL metrics

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / "metrics"
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob("*.json"))

    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # Sort by modification time (newest first)
        metrics_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = metrics_files[0]

        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)

        try:
            with open(latest_metrics_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            metadata = data.get("metadata", {})
            metrics = data.get("metrics", {})

            # ------------------------------------------------------------------
            # METADATA
            # ------------------------------------------------------------------
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")

            if "operation" in metadata:
                op = metadata["operation"]
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")

            # ------------------------------------------------------------------
            # KS METRICS
            # ------------------------------------------------------------------
            if "ks" in metrics:
                print("\n📈 KOLMOGOROV–SMIRNOV TEST:")
                ks = metrics["ks"]

                for key in [
                    "ks_statistic",
                    "p_value",
                    "max_difference",
                    "effect_size",
                    "statistical_significance",
                    "normalization_applied",
                    "confidence_level",
                    "alpha",
                ]:
                    if key in ks:
                        value = ks[key]
                        label = key.replace("_", " ").title()

                        if isinstance(value, float):
                            print(f"   {label:<30}: {value:.6f}")
                        else:
                            print(f"   {label:<30}: {value}")

                if "confidence_interval" in ks:
                    print(
                        f"   {'Confidence Interval':<30}: "
                        f"{ks['confidence_interval']}"
                    )

                if "interpretation" in ks:
                    print(f"\n   Interpretation:")
                    print(f"     {ks['interpretation']}")

            # ------------------------------------------------------------------
            # KL METRICS
            # ------------------------------------------------------------------
            if "kl" in metrics:
                print("\n📉 KULLBACK–LEIBLER DIVERGENCE:")
                kl = metrics["kl"]

                for key in [
                    "kl_divergence",
                    "kl_divergence_bits",
                    "jensen_shannon_distance",
                    "effect_size",
                    "statistical_significance",
                    "smoothing_applied",
                    "normalization_applied",
                    "confidence_level",
                ]:
                    if key in kl:
                        value = kl[key]
                        label = key.replace("_", " ").title()

                        if isinstance(value, float):
                            print(f"   {label:<30}: {value:.6f}")
                        else:
                            print(f"   {label:<30}: {value}")

                if "confidence_interval" in kl:
                    print(
                        f"   {'Confidence Interval':<30}: "
                        f"{kl['confidence_interval']}"
                    )

                if "interpretation" in kl:
                    print(f"\n   Interpretation:")
                    print(f"     {kl['interpretation']}")

            # ------------------------------------------------------------------
            # PERFORMANCE
            # ------------------------------------------------------------------
            print("\n⚡ PERFORMANCE:")

            perf_keys = [
                ("duration_seconds", "Duration (s)"),
                ("records_processed", "Records Processed"),
                ("records_per_second", "Records / Second"),
                ("sample_size", "Sample Size"),
                ("total_original_records", "Original Records"),
                ("total_transformed_records", "Transformed Records"),
            ]

            for key, label in perf_keys:
                if key in metrics:
                    value = metrics[key]
                    if value is None:
                        print(f"   {label:<25}: N/A")
                    elif isinstance(value, float):
                        print(f"   {label:<25}: {value:.4f}")
                    else:
                        print(f"   {label:<25}: {value}")

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. VISUALIZATIONS (NEWEST BATCH)
print("\n\n2️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load original and transformed data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure fidelity metrics operation (KS test, KL divergence)  
✅ Execute using operation.execute()  
✅ Analyze statistical results and interpret fidelity  

**Key takeaways:**
- **KS Test**: Measures if two distributions differ significantly
  - Low p-value (< 0.05) = distributions differ significantly
  - High p-value (≥ 0.05) = distributions are similar (good fidelity)
- **KL Divergence**: Measures how much one distribution diverges from another
  - Lower values = better fidelity
  - Values close to 0 = distributions are very similar
- Effect sizes help interpret practical significance
- Confidence intervals provide uncertainty estimates
- All artifacts saved in structured directories

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_fidelity_advanced.ipynb`** includes:
- Additional fidelity metrics (Wasserstein distance, Chi-square test)
- Per-column fidelity analysis
- Custom metric parameters and thresholds
- Sampling strategies for large datasets
- Column mapping for renamed fields
- Performance optimization with Dask
- Multiple confidence levels comparison

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 📊 [Fidelity Metrics Guide](../../docs/fidelity_metrics.md)